# F01: Bronze/Silver/Gold Medallion Architecture

Build a complete medallion lakehouse pipeline with synthetic data -- from messy landing zone to curated gold layer.

## What You'll Learn
- Generate realistic retail data at medium scale with Spindle
- **Bronze:** Write raw Parquet files with chaos injection (messy, real-world data)
- **Silver:** Clean and validate data using `ValidationGates`
- **Gold:** Transform to a star schema with `StarSchemaTransform`
- Organize files into a proper medallion folder structure

## Prerequisites
- Python 3.10+
- `sqllocks-spindle` installed
- Basic understanding of the medallion architecture pattern

## Time Estimate
~20 minutes

In [ ]:
# %pip install sqllocks-spindle

## Step 1: Generate Retail Data at Medium Scale

**What we're about to do:** Generate a full retail dataset -- customers, products, stores, orders, and order lines -- at medium scale (~5,000 customers, ~20,000 orders).

**Why this matters:** Real medallion pipelines ingest messy, upstream data. Spindle gives us realistic relational data with proper FK relationships so we can build and test each tier.

**What to expect:** A `GenerationResult` with 5+ tables and ~50,000 total rows. Generation takes a few seconds.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="medium", seed=42)

print(result.summary())
print(f"\nFK integrity errors: {len(result.verify_integrity())}")

## Step 2: Bronze -- Write Raw Parquet with Chaos Injection

**What we're about to do:** Write the generated data to a `bronze/` landing zone as raw Parquet files. Then we'll inject chaos (nulls, type mismatches, orphan FKs) to simulate real-world messy upstream data.

**Why this matters:** The bronze layer should contain data exactly as it arrived -- warts and all. Testing your pipeline against clean data gives you false confidence. The `ChaosEngine` lets you control exactly what kinds of issues to inject.

**Key concepts:**
- `ChaosConfig` controls intensity (`calm`, `moderate`, `stormy`, `hurricane`) and which categories fire (value, schema, file, referential, temporal, volume)
- `to_parquet()` writes one `.parquet` file per table

In [ ]:
from pathlib import Path
from sqllocks_spindle.chaos.config import ChaosConfig
from sqllocks_spindle.chaos.engine import ChaosEngine

# Set up medallion directory structure
base_dir = Path("medallion_demo")
bronze_dir = base_dir / "bronze" / "retail"
silver_dir = base_dir / "silver" / "retail"
gold_dir   = base_dir / "gold" / "retail"

for d in [bronze_dir, silver_dir, gold_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Write clean data first as raw Parquet (the "as-generated" landing)
paths = result.to_parquet(bronze_dir)
print(f"Bronze: wrote {len(paths)} Parquet files to {bronze_dir}/")
for p in paths:
    print(f"  {p.name} ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Now inject chaos into the bronze data to simulate messy upstream files
chaos_cfg = ChaosConfig(
    enabled=True,
    intensity="moderate",        # 1.0x multiplier on base probabilities
    seed=99,
    warmup_days=0,
    chaos_start_day=0,
    categories={
        "value":       {"enabled": True, "weight": 0.20},   # nulls, wrong types, out-of-range
        "schema":      {"enabled": True, "weight": 0.10},   # extra/missing columns
        "referential": {"enabled": True, "weight": 0.15},   # orphan FK values
        "temporal":    {"enabled": False, "weight": 0.0},
        "file":        {"enabled": False, "weight": 0.0},
        "volume":      {"enabled": False, "weight": 0.0},
    },
)
chaos = ChaosEngine(chaos_cfg)

# Apply value and schema chaos to each table
import pandas as pd

bronze_tables = {}
for table_name, df in result.tables.items():
    corrupted = chaos.corrupt_dataframe(df.copy(), day=10)
    corrupted = chaos.drift_schema(corrupted, day=10)
    bronze_tables[table_name] = corrupted
    null_count = corrupted.isna().sum().sum()
    print(f"  {table_name}: {len(corrupted)} rows, {null_count} null values injected")

# Inject referential chaos across the full table set
bronze_tables = chaos.inject_referential_chaos(bronze_tables, day=10)
print("\nBronze layer chaos injection complete.")

## Step 3: Silver -- Clean and Validate with ValidationGates

**What we're about to do:** Run the corrupted bronze data through Spindle's `GateRunner` with all 8 built-in validation gates. Then we'll clean up the issues and quarantine records that can't be fixed.

**Why this matters:** The silver layer is your "validated, conformed" tier. Every record that makes it past the gates is trustworthy. Records that fail go to quarantine for investigation.

**Key concepts:**
- `GateRunner` runs all gates (referential integrity, schema conformance, null constraints, unique constraints, range constraints, temporal consistency, file format, schema drift)
- `QuarantineManager` isolates bad records with full metadata for debugging

In [ ]:
from sqllocks_spindle.validation.gates import (
    GateRunner, ValidationContext, GateResult
)

# Run validation gates against the bronze (corrupted) data
context = ValidationContext(
    tables=bronze_tables,
    schema=result.schema,
)

runner = GateRunner(gates=[
    "referential_integrity",
    "schema_conformance",
    "null_constraint",
    "unique_constraint",
])
gate_results = runner.run_all(context)

# Print gate summary
summary = GateRunner.summary(gate_results)
print(f"Gates run:   {summary['total_gates']}")
print(f"Passed:      {summary['passed']}")
print(f"Failed:      {summary['failed']}")
print(f"Total errors: {summary['total_errors']}")
print()

for gr in gate_results:
    status = "PASS" if gr.passed else "FAIL"
    print(f"  [{status}] {gr.gate_name}: {len(gr.errors)} errors, {len(gr.warnings)} warnings")
    for err in gr.errors[:3]:  # Show first 3 errors per gate
        print(f"         {err}")

In [ ]:
from sqllocks_spindle.validation.quarantine import QuarantineManager

# Quarantine tables that failed validation, then use clean originals for silver
quarantine_dir = base_dir / "quarantine"
qm = QuarantineManager(domain="retail")

# For each table, compare bronze (corrupted) vs original and quarantine bad rows
silver_tables = {}
for table_name, bronze_df in bronze_tables.items():
    original_df = result.tables[table_name]
    original_cols = set(original_df.columns)
    bronze_cols = set(bronze_df.columns)

    # Drop any chaos-injected extra columns to restore schema conformance
    extra_cols = bronze_cols - original_cols
    clean_df = bronze_df.drop(columns=list(extra_cols), errors="ignore")

    # Drop rows with null PKs (unrecoverable)
    pk_cols = result.schema.tables[table_name].primary_key
    if pk_cols:
        bad_mask = clean_df[pk_cols].isna().any(axis=1)
        if bad_mask.sum() > 0:
            bad_rows = clean_df[bad_mask]
            qm.quarantine_dataframe(
                bad_rows, quarantine_dir, run_id="bronze_v1",
                table_name=table_name, reason="Null primary key",
                gate_name="null_constraint",
            )
            clean_df = clean_df[~bad_mask]

    silver_tables[table_name] = clean_df
    print(f"  {table_name}: {len(bronze_df)} bronze -> {len(clean_df)} silver rows")

# Write silver to Parquet
for name, df in silver_tables.items():
    df.to_parquet(silver_dir / f"{name}.parquet", index=False)

print(f"\nSilver: wrote {len(silver_tables)} cleaned tables to {silver_dir}/")

## Step 4: Gold -- Transform to Star Schema

**What we're about to do:** Use `StarSchemaTransform` to convert the cleaned silver tables into a proper dimensional model with surrogate keys, a date dimension, and fact/dim separation.

**Why this matters:** The gold layer is optimized for analytics. Dimension tables are denormalized, fact tables carry surrogate keys, and a `dim_date` table enables time intelligence in Power BI.

**What to expect:** A `StarSchemaResult` with `dim_*` and `fact_*` tables, plus an auto-generated `dim_date`.

In [ ]:
from sqllocks_spindle import (
    StarSchemaTransform, StarSchemaMap, DimSpec, FactSpec
)

# Define the star schema mapping for retail domain
schema_map = StarSchemaMap(
    dims={
        "dim_customer": DimSpec(
            source="customer", sk="sk_customer", nk="customer_id"
        ),
        "dim_product": DimSpec(
            source="product", sk="sk_product", nk="product_id"
        ),
        "dim_store": DimSpec(
            source="store", sk="sk_store", nk="store_id"
        ),
    },
    facts={
        "fact_order": FactSpec(
            primary="order",
            fk_map={
                "customer_id": "dim_customer",
                "store_id": "dim_store",
            },
            date_cols=["order_date"],
        ),
    },
    generate_date_dim=True,
)

# Run the transform on silver (clean) data
transform = StarSchemaTransform()
star = transform.transform(silver_tables, schema_map)

print(star.summary())

In [ ]:
# Write gold star schema tables to Parquet
gold_tables = star.all_tables()
for name, df in gold_tables.items():
    df.to_parquet(gold_dir / f"{name}.parquet", index=False)
    print(f"  {name}: {len(df):,} rows x {len(df.columns)} cols")

print(f"\nGold: wrote {len(gold_tables)} star schema tables to {gold_dir}/")

# Show the folder structure
print("\n--- Medallion Directory Structure ---")
for tier in ["bronze", "silver", "gold", "quarantine"]:
    tier_path = base_dir / tier
    if tier_path.exists():
        files = list(tier_path.rglob("*"))
        data_files = [f for f in files if f.is_file() and not f.name.startswith(".")]
        print(f"  {tier}/  ({len(data_files)} files)")
        for f in sorted(data_files)[:5]:
            print(f"    {f.relative_to(base_dir)}")

## What's Next?

You've built a complete Bronze/Silver/Gold pipeline with realistic data quality issues and validation.

- **F05: Chaos Pipeline Testing** -- Go deeper with the `ChaosEngine`: test all 6 chaos categories, tune intensity presets, and build a full quarantine workflow
- **F06: Semantic Model Builder** -- Generate a `.bim` file from your gold star schema so Power BI relationships and measures are pre-wired
- **F02: Warehouse Dimensional Load** -- Load the gold star schema directly into a Fabric Warehouse via T-SQL